# Topic 2 - Next-Token Prediction

This notebook makes the autoregressive loop visible: `context -> distribution -> select -> append -> repeat`. It uses a tiny character-level model, so no download is needed.

Training versus inference, in one line: during training the model learns to predict the next token from preceding tokens; during inference it uses those patterns to generate tokens sequentially.

In [2]:
from collections import Counter, defaultdict
import math
import random

random.seed(7)
corpus = ('''the cat sat on the mat. the cat slept on the mat. \n
          the dog sat by the door. the dog slept by the door.\n
          a small model predicts one token at a time.\n
          a model can continue a familiar pattern. ''')
tokens = sorted(set(corpus))
counts = defaultdict(Counter)
for current_token, next_token in zip(corpus, corpus[1:]):
    counts[current_token][next_token] += 1
print(f'Corpus characters: {len(corpus):,}')
print(f'Vocabulary size: {len(tokens)}')

Corpus characters: 221
Vocabulary size: 23


## 1. Inspect a next-token distribution

The model assigns a probability to every possible next token. The distribution is conditioned on the current context.

In [4]:
def next_token_distribution(context):
    if not context:
        raise ValueError('context must contain at least one token')
    row = counts[context[-1]]
    denominator = sum(row.values()) + len(tokens)
    return {token: (row[token] + 1) / denominator for token in tokens}

def show_predictions(context, limit=8):
    distribution = next_token_distribution(context)
    print(f'Context: {context!r}')
    for token, probability in sorted(distribution.items(), key=lambda item: item[1], reverse=True)[:limit]:
        print(f'  {token!r:>4} -> {probability:.3f}')

assert math.isclose(sum(next_token_distribution('t').values()), 1.0)
show_predictions('the cat sat on th')
print()
show_predictions('the dog sat by th')

Context: 'the cat sat on th'
   'e' -> 0.290
  '\n' -> 0.032
   ' ' -> 0.032
   '.' -> 0.032
   'a' -> 0.032
   'b' -> 0.032
   'c' -> 0.032
   'd' -> 0.032

Context: 'the dog sat by th'
   'e' -> 0.290
  '\n' -> 0.032
   ' ' -> 0.032
   '.' -> 0.032
   'a' -> 0.032
   'b' -> 0.032
   'c' -> 0.032
   'd' -> 0.032


## 2. Predict, select, append, repeat

Greedy decoding chooses the highest-probability token. Sampling chooses according to the distribution. The loop is identical; only selection differs.

In [5]:
def greedy_select(distribution):
    return max(distribution, key=distribution.get)

def sample_select(distribution, rng=random):
    choices = list(distribution)
    return rng.choices(choices, weights=[distribution[token] for token in choices], k=1)[0]

def generate(prompt, steps, select_token):
    context, log = prompt, []
    for step in range(1, steps + 1):
        distribution = next_token_distribution(context)
        token = select_token(distribution)
        before = context
        context += token
        log.append((step, before, token, context, max(distribution.values())))
    return context, log

text, log = generate('the cat sat on th', 30, greedy_select)
print(text)
for step, before, token, after, top_probability in log[:12]:
    print(f'Step {step:>2}: {before!r} -> {token!r} -> {after!r} (top p={top_probability:.3f})')

the cat sat on the                             
Step  1: 'the cat sat on th' -> 'e' -> 'the cat sat on the' (top p=0.290)
Step  2: 'the cat sat on the' -> ' ' -> 'the cat sat on the ' (top p=0.268)
Step  3: 'the cat sat on the ' -> ' ' -> 'the cat sat on the  ' (top p=0.311)
Step  4: 'the cat sat on the  ' -> ' ' -> 'the cat sat on the   ' (top p=0.311)
Step  5: 'the cat sat on the   ' -> ' ' -> 'the cat sat on the    ' (top p=0.311)
Step  6: 'the cat sat on the    ' -> ' ' -> 'the cat sat on the     ' (top p=0.311)
Step  7: 'the cat sat on the     ' -> ' ' -> 'the cat sat on the      ' (top p=0.311)
Step  8: 'the cat sat on the      ' -> ' ' -> 'the cat sat on the       ' (top p=0.311)
Step  9: 'the cat sat on the       ' -> ' ' -> 'the cat sat on the        ' (top p=0.311)
Step 10: 'the cat sat on the        ' -> ' ' -> 'the cat sat on the         ' (top p=0.311)
Step 11: 'the cat sat on the         ' -> ' ' -> 'the cat sat on the          ' (top p=0.311)
Step 12: 'the cat sat on the

## 3. Sampling and trajectory drift

A sampled token becomes part of the next context. One different choice can therefore change every later distribution.

In [6]:
for seed in [1, 2, 3]:
    rng = random.Random(seed)
    text, _ = generate('the cat sat on th', 45, lambda distribution: sample_select(distribution, rng))
    print(f'Run {seed}: {text}')

Run 1: the cat sat on thbupcheloa sgp
dodut  dufbh
 bkcabid
rimays.at
Run 2: the cat sat on thuu  snmdnkkakfoyukgb

e.cthga
a dymatoosopfyy
Run 3: the cat sat on thehekm 
rbcygrgl lskon pmd
shmsmtfrfut   thidk


## Three behaviors, one mechanism

- **Confident wrongness:** high probability means plausible under learned patterns, not guaranteed true in the world.
- **Verbosity:** answer-like continuations can make explanations and examples likely, so the loop continues.
- **Losing the thread:** a generated token changes the context; a shifted trajectory conditions later predictions.

Exercises: change the corpus; add temperature; add a period stop condition; try a false premise; explain an early error using the generation log.

> An autoregressive language model generates a probability distribution over the next token given the current context, selects a token, appends it, and repeats until generation stops.